In [1]:
import os, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_res
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_eff
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score

BASE = '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri'
TRAIN_DIR = os.path.join(BASE, 'Training')
TEST_DIR  = os.path.join(BASE, 'Testing')

def mc_model(name):
    inp = layers.Input(shape=(224, 224, 3))
    if name == 'resnet50':
        base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
        pre = pre_res
    else:
        base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)
        pre = pre_eff
    
    base.trainable = True
    for L in base.layers[:-50]: L.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x, training=True)  # MC Dropout
    out = layers.Dense(4, activation='softmax')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return m, pre

# Train ResNet50 MC (seed 42 only, ~15 min)
for model_name in ['resnet50', 'efficientnetb0']:
    print(f"\n{'='*50}\nMC Dropout: {model_name.upper()}\n{'='*50}")
    model, pre = mc_model(model_name)
    
    tr_aug = ImageDataGenerator(preprocessing_function=pre, rotation_range=20,
        width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
        horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)
    tr = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
        class_mode='categorical', subset='training', seed=42)
    val = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
        class_mode='categorical', subset='validation', shuffle=False, seed=42)
    te = ImageDataGenerator(preprocessing_function=pre).flow_from_directory(
        TEST_DIR, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)
    
    cb = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]
    model.fit(tr, validation_data=val, epochs=30, callbacks=cb, verbose=1)
    
    te.reset(); loss, acc = model.evaluate(te, verbose=0)
    te.reset(); yp = np.argmax(model.predict(te, verbose=0), axis=1)
    yt = te.classes
    print(f"{model_name} MC — Accuracy: {acc*100:.2f}% | F1: {f1_score(yt, yp, average='weighted')*100:.2f}%")


MC Dropout: RESNET50


I0000 00:00:1785659095.766633      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785659095.769626      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
Epoch 1/30


I0000 00:00:1785659134.379968     135 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


140/140 ━━━━━━━━━━━━━━━━━━━━ 144s 818ms/step - accuracy: 0.8239 - loss: 0.5200 - val_accuracy: 0.9054 - val_loss: 0.2913 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 84s 596ms/step - accuracy: 0.9252 - loss: 0.2200 - val_accuracy: 0.9375 - val_loss: 0.1833 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 83s 592ms/step - accuracy: 0.9464 - loss: 0.1525 - val_accuracy: 0.9402 - val_loss: 0.2091 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 82s 585ms/step - accuracy: 0.9560 - loss: 0.1420 - val_accuracy: 0.9580 - val_loss: 0.1805 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 83s 592ms/step - accuracy: 0.9661 - loss: 0.1084 - val_accuracy: 0.9625 - val_loss: 0.1673 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 83s 592ms/step - accuracy: 0.9710 - loss: 0.0878 - val_accuracy: 0.9518 - val_loss: 0.1625 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 82s 584ms/step -

2026-08-02 08:57:13.510105: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-02 08:57:13.654536: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-02 08:57:14.016606: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-02 08:57:14.158210: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-02 08:57:15.010887: E external/local_xla/xla/stream_

140/140 ━━━━━━━━━━━━━━━━━━━━ 125s 621ms/step - accuracy: 0.6676 - loss: 0.9432 - val_accuracy: 0.8741 - val_loss: 0.4343 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 557ms/step - accuracy: 0.8279 - loss: 0.4878 - val_accuracy: 0.8902 - val_loss: 0.2908 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 558ms/step - accuracy: 0.8739 - loss: 0.3443 - val_accuracy: 0.9232 - val_loss: 0.2271 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 559ms/step - accuracy: 0.8908 - loss: 0.3090 - val_accuracy: 0.9330 - val_loss: 0.1856 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 554ms/step - accuracy: 0.9141 - loss: 0.2285 - val_accuracy: 0.9330 - val_loss: 0.1727 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 558ms/step - accuracy: 0.9250 - loss: 0.2097 - val_accuracy: 0.9420 - val_loss: 0.1610 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 80s 574ms/step -

In [ ]:
import os

print("=== All datasets in /kaggle/input ===\n")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    if level == 0:
        for d in sorted(dirs):
            path = os.path.join(root, d)
            subdirs = [s for s in os.listdir(path) if os.path.isdir(os.path.join(path, s))]
            print(f"📁 {d}")
            if subdirs:
                print(f"   └── Subfolders: {subdirs[:5]}")
    elif level == 1:
        # Show files in first-level dirs
        for f in sorted(files)[:5]:
            print(f"   📄 {f}")
        if len(files) > 5:
            print(f"   ... and {len(files)-5} more files")